为实现从理论框架到工程落地的有效转化，以下将前期梳理的方法论与 `sbi` 核心代码进行严格对齐。此教学设计剔除了原教程的口语化赘述，将其重构为标准的算法实现规程（Standard Operating Procedure），以展示如何构建基于神经后验估计（NPE）的推断管线。

---

### 一、 参数空间映射与先验构筑 (Parameter Mapping & Prior Definition)

**物理逻辑**：推断管线的首要任务是确立底层参数空间 $\theta$ 与观测空间 $x$ 之间的前向映射算子，并界定参数的物理容许域（即先验分布 $p(\theta)$）。

**代码实现**：
此处引入一个简化的解析函数作为前向仿真器（Simulator）的替代。在黑洞光谱学工程中，此函数将被替换为引力波 Ringdown 信号生成器（如高斯白噪声与指数阻尼正弦波的线性叠加，或含 $\alpha$ 偏差的非克尔波形生成工具）。

```python
import torch
from sbi.utils import BoxUniform

# 固定随机种子以确保实验可复现性
_ = torch.manual_seed(0)

num_dim = 3

# 1. 定义前向仿真器 (映射算子)
def simulator(theta):
    # 线性高斯映射：充当复杂物理模型的占位符
    # 模拟观测过程中的系统偏移 (+1.0) 与仪器噪声 (Gaussian noise)
    return theta + 1.0 + torch.randn_like(theta) * 0.1

# 2. 定义联合先验分布
# 构建一个三维的均匀先验，边界严格限制在 [-2, 2] 之间
prior = BoxUniform(low=-2 * torch.ones(num_dim), high=2 * torch.ones(num_dim))

```

---

### 二、 仿真数据集生成 (Simulation Dataset Generation)

**物理逻辑**：通过蒙特卡洛方法从先验分布中抽取参数样本，并驱动仿真器生成对应的观测数据，从而构建用于训练神经网络的配对数据集 $\mathcal{D} = \{(\theta_i, x_i)\}_{i=1}^N$。

**代码实现**：

```python
num_simulations = 2000

# 从先验中抽取参数矩阵
theta = prior.sample((num_simulations,))

# 执行前向仿真，生成对应的观测张量
x = simulator(theta)

# 验证张量维度是否对齐
print("theta.shape:", theta.shape) # 预期输出: torch.Size([2000, 3])
print("x.shape:", x.shape)         # 预期输出: torch.Size([2000, 3])

```

---

### 三、 密度估计器实例化与网络训练 (Density Estimator Training)

**物理逻辑**：实例化神经后验估计（NPE）架构。该架构利用 Normalizing Flows 学习 $\theta$ 与 $x$ 之间的非线性流形映射。训练过程旨在最大化验证集上的对数似然。

**代码实现**：

```python
from sbi.inference import NPE

# 1. 实例化推理对象，注入先验约束
inference = NPE(prior=prior)

# 2. 注入仿真数据集
inference = inference.append_simulations(theta, x)

# 3. 启动网络训练循环
# 返回的 density_estimator 是完成权重优化的流模型
density_estimator = inference.train()

```

---

### 四、 摊销推断的执行与后验采样 (Amortized Inference & Posterior Sampling)

**物理逻辑**：网络收敛后，推断管线具备了“摊销（Amortization）”属性。给定具体的观测数据张量 $x_{obs}$，模型可通过单次前向传播，通过拒收采样（Rejection Sampling）过滤先验边界外的点，直接输出目标后验分布 $p(\theta|x_{obs})$ 的样本。

**代码实现**：

```python
# 1. 构建后验评估器 (自动包含对先验边界的截断机制)
posterior = inference.build_posterior()

# 2. 定义给定的观测数据 (模拟真实探测器接收到的信号)
x_obs = torch.as_tensor([0.8, 0.6, 0.4])

# 3. 执行条件后验采样
# 在给定 x_obs 的条件下，抽取 10000 个参数后验样本
posterior_samples = posterior.sample((10000,), x=x_obs)

```

---

### 五、 结果可视化与物理诊断 (Visualization & Diagnostics)

**物理逻辑**：推断结果的科学严谨性需要双重验证：

1. **参数空间协方差分析**：评估各维度参数的推断精度与退化关系。
2. **后验预测检验 (PPC)**：将推断出的后验样本重新代入物理机制（仿真器），验证其生成的预测信号与初始观测信号的统计距离。

**代码实现**：

```python
from sbi.analysis import pairplot

# 1. 绘制后验样本的边缘分布与二维联合分布 (角图)
_ = pairplot(
    posterior_samples,
    limits=[[-2, 2], [-2, 2], [-2, 2]],
    figsize=(5, 5),
    labels=[r"$\theta_1$", r"$\theta_2$", r"$\theta_3$"]
)

# 2. 执行后验预测检验 (Posterior Predictive Checks)
# 将后验样本重新代入仿真器
x_predictive = simulator(posterior_samples)

# 计算预测观测值的期望并与真实观测值比对
print("Posterior predictives (Expected):", torch.mean(x_predictive, axis=0))
print("Observation (Actual):", x_obs)
# 若两者高度重合，则证明推断管线未出现严重的模型错误设定 (Misspecification)

```

---

**后续步骤建议：**
为实现课题工程的真实物理映射，您是否需要我将上述代码中的 `simulator(theta)` 函数，替换为一个包含指数阻尼项与频率偏差参数 $\alpha$ 的一维时间序列 Ringdown 波形生成器代码框架？
